In [ ]:
import os

# Set the visible devices to only GPU 2
# os.environ["CUDA_VISIBLE_DEVICES"] = "3"

# Verify CUDA device visibility in PyTorch
import torch
print("Available CUDA devices:", torch.cuda.device_count())
print("Using device:", torch.cuda.current_device())

import numpy as np
import DP as dp
import pandas as pd
import numpy as np
import process_edited as pce
from tqdm import tqdm
import itertools
import pickle

In [ ]:
hiv_dt = pd.read_csv("~/Documents/TimeAutoDiff/Dataset/hiv.csv.gz")
## train split
unique_id = hiv_dt['patient_id'].unique()
np.random.seed(123123)
unique_id = np.random.choice(unique_id, size=int(len(unique_id)*0.85), replace=False)

print(f'Training size: {len(unique_id)}')
test_df = hiv_dt.loc[~hiv_dt.patient_id.isin(unique_id)].reset_index(drop=True)
real_df = hiv_dt.loc[hiv_dt.patient_id.isin(unique_id)].reset_index(drop=True)

print(real_df.shape)

real_df1 = real_df.drop('date', axis=1)

In [ ]:
### For synteg

## Label:
real_lb = real_df.groupby('patient_id').first()[['enrol_d','age','male_y','center','mode']].reset_index()
real_lb.reset_index()
enrol_d = np.repeat(pd.cut(real_lb.enrol_d,bins=30,labels=False).values[:, np.newaxis],120,axis=1)
age = np.repeat(pd.cut(real_lb.age,bins=30,labels=False).values[:, np.newaxis],120,axis=1)
male_y = real_lb.male_y.astype('int').values[:, np.newaxis]
male_y = np.repeat(male_y,120,axis=1)
center = np.repeat(real_lb.center.astype('int').values[:, np.newaxis],120,axis=1)
center2site = {3: 2, 4: 2, 1: 1, 2: 2, 5: 3, 7: 5, 6: 5, 9: 6, 8: 4}
site = np.repeat(real_lb.center.map(center2site).astype('int').values[:, np.newaxis],120,axis=1)
mode = np.repeat(real_lb['mode'].astype('int').values[:, np.newaxis],120,axis=1)
np.save("./data/datasets/synteg/birth",enrol_d)
np.save("./data/datasets/synteg/age.npy",age)
np.save("./data/datasets/synteg/gender.npy",male_y)
np.save("./data/datasets/synteg/center.npy",center-1)
np.save("./data/datasets/synteg/country.npy",site-1)
np.save("./data/datasets/synteg/mode.npy",mode)

print(f"enrol_d: {len(np.unique(enrol_d))}")
print(f"age: {len(np.unique(age))}")
print(f"male_y: {len(np.unique(male_y))}")
print(f"center: {len(np.unique(center))}")
print(f"site: {len(np.unique(site))}")
print(f"mode: {len(np.unique(mode))}")

## Visits:
real_df = real_df.drop(columns=['enrol_d','age','male_y','center','mode']).groupby('patient_id', group_keys=False).apply(lambda x: x.iloc[1:])


gap = pd.cut(real_df.pop('gap').fillna(0),bins=30,labels=False)
gap = pd.get_dummies(gap) * 1
gap["patient_id"] = real_df.patient_id.copy()
gap['date'] = 0
gap, _, _ = dp.partition_multi_seq(gap,1,'patient_id',120)
gap = np.argmax(gap,axis=-1)
np.save("./data/datasets/synteg/interval.npy",gap.numpy())


In [ ]:

## Continuous
cd4 = pd.get_dummies(pd.cut(real_df.pop('cd4_v'),bins=30,labels=False),prefix='cd4') * 1
rna = pd.get_dummies(pd.cut(real_df.pop('rna_v'),bins=30,labels=False),prefix='rna') * 1
weight = pd.get_dummies(pd.cut(real_df.pop('weight'),bins=30,labels=False),prefix='weight') * 1
height = pd.get_dummies(pd.cut(real_df.pop('height'),bins=30,labels=False),prefix='height') * 1


real_df = real_df.drop(real_df.drop('date',axis=1).columns[real_df.drop(columns="date").std()==0],axis=1)
real_df = pd.concat([real_df,cd4,rna,weight,height],axis=1)


In [ ]:
# Extract patient IDs and data excluding the patient_id column
if 'date' in real_df.columns: real_df.drop('date',axis=1,inplace=True)
code_map = real_df.drop(columns=['patient_id']).columns
pickle.dump(code_map, open("./data/datasets/synteg/code_map.pkl","wb"))
patient_ids = real_df['patient_id'].values
data = real_df.drop(columns=['patient_id']).values * 1

# Get the number of patients and the number of features (excluding patient_id)
num_patients = len(patient_ids)
num_features = data.shape[1]

# Create a dictionary to store patient visit data
patient_data_dict = {patient_id: [] for patient_id in np.unique(patient_ids)}

# Populate the patient_data_dict with visit data
for i in tqdm(range(num_patients),total=num_patients):
    patient_id = patient_ids[i]
    visit_data = data[i]
    visit_code = np.where(visit_data == 1)[0] 
    patient_data_dict[patient_id].append(visit_code.tolist())

# Convert dictionary values to list of lists format
code = list(patient_data_dict.values())

In [ ]:

res_c = []
length = []

max_length = 120
print("max length: %d" % max_length)

max_len = max(len(y) for x in code for y in x) + 1
print('max_len', max_len, flush=True)

code_map = real_df.drop(columns=['patient_id']).columns
code_map = {j:i for i,j in zip(code_map, np.arange(len(code_map)))}
code_last = len(code_map)
print('num code', code_last, flush=True)

for c in tqdm(code):
    # a = [a] * len(i)
    c = c[:max_length]
    c = [np.concatenate((episode_c, -np.ones(max_len - len(episode_c))), axis=0) for episode_c in c] ## each day has the same number of codes after padding
    length.append(len(c)) # why not 0?
    c[len(c)-1][-1] = code_last  
    if len(c) < max_length:
        c = np.concatenate((c, -np.ones((max_length-len(c), max_len))),axis=0) ## each patient has the same number of days after padding
    else: c = np.array(c)
    res_c.append(c.astype(int))

np.save("./data/datasets/synteg/code.npy", np.array(res_c, dtype='int32'))
np.save("./data/datasets/synteg/length.npy", np.array(length, dtype='int32'))